In [ ]:
from google.colab import files
uploaded = files.upload()

Saving patients.csv to patients.csv
Saving billing.csv to billing.csv
Saving visits.csv to visits.csv


In [ ]:
#build features

import pandas as pd
import numpy as np

# Load CSVs
patients_df = pd.read_csv("patients.csv")
visits_df   = pd.read_csv("visits.csv")
billing_df  = pd.read_csv("billing.csv")

# Merge visits with patients
visits_patients = visits_df.merge(patients_df, on="patient_id", how="left")

# Merge billing
full_df = visits_patients.merge(billing_df, on="visit_id", how="left")


# feature engineering

# Visit frequency per patient
visit_freq = visits_df.groupby('patient_id').size().reset_index(name='visit_frequency')
full_df = full_df.merge(visit_freq, on='patient_id', how='left')

# Avg length of stay per patient
avg_los = visits_df.groupby('patient_id')['length_of_stay_hours'].mean().reset_index(name='avg_length_of_stay')
full_df = full_df.merge(avg_los, on='patient_id', how='left')

# Provider rejection rate
rejection_rate = full_df.groupby('insurance_provider')['claim_status'].apply(
    lambda x: (x=='Rejected').mean()
).reset_index(name='provider_rejection_rate')
full_df = full_df.merge(rejection_rate, on='insurance_provider', how='left')

# Days since registration
full_df['registration_date'] = pd.to_datetime(full_df['registration_date'])
full_df['visit_date'] = pd.to_datetime(full_df['visit_date'])
full_df['days_since_registration'] = (full_df['visit_date'] - full_df['registration_date']).dt.days

# Time-based features
full_df['visit_month'] = full_df['visit_date'].dt.month
full_df['visit_weekday'] = full_df['visit_date'].dt.weekday
full_df['visit_hour'] = full_df['visit_date'].dt.hour

# save modelling dataset
full_df.to_csv("model_table.csv", index=False)

In [ ]:
# risk model

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import json

In [ ]:
# Load dataset
full_df = pd.read_csv("model_table.csv")

# Model A: Visit Risk Classification

print("=== Model A: Visit Risk Classification ===")
full_df = full_df.sort_values('visit_date')  # Time-based split
train_size = int(len(full_df) * 0.8)
train_df = full_df.iloc[:train_size]
test_df = full_df.iloc[train_size:]

target = 'risk_score'
features = [
    'age', 'gender', 'chronic_flag', 'city', 'department', 'visit_type',
    'length_of_stay_hours', 'visit_frequency', 'avg_length_of_stay',
    'provider_rejection_rate', 'days_since_registration',
    'visit_month', 'visit_weekday'
]
X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# 2a️ Preprocessing
numeric_features = ['age', 'length_of_stay_hours', 'visit_frequency',
                    'avg_length_of_stay', 'provider_rejection_rate',
                    'days_since_registration', 'visit_month', 'visit_weekday']
categorical_features = ['gender', 'city', 'department', 'visit_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# 2b️ Baseline Model

lr_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', LogisticRegression(max_iter=1000))])
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
print("Baseline Logistic Regression Report")
print(classification_report(y_test, y_pred))

# 2c️ Advanced Model

rf_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))])
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Report")
print(classification_report(y_test, y_pred_rf))

# Save model
joblib.dump(rf_model, 'risk_model.pkl')

=== Model A: Visit Risk Classification ===
Baseline Logistic Regression Report


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

        High       0.00      0.00      0.00      1023
         Low       0.50      1.00      0.66      2480
      Medium       1.00      0.00      0.00      1497

    accuracy                           0.50      5000
   macro avg       0.50      0.33      0.22      5000
weighted avg       0.55      0.50      0.33      5000

Random Forest Report
              precision    recall  f1-score   support

        High       0.16      0.01      0.01      1023
         Low       0.50      0.87      0.64      2480
      Medium       0.31      0.14      0.19      1497

    accuracy                           0.47      5000
   macro avg       0.32      0.34      0.28      5000
weighted avg       0.37      0.47      0.37      5000



['risk_model.pkl']

In [ ]:
#claim model

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

In [ ]:
# Load dataset

full_df = pd.read_csv("model_table.csv")  # Output from Phase 2

# Model B: Claim Outcome Classification

print("\n=== Model B: Claim Outcome Classification ===")

# Time-based train-test split
full_df = full_df.sort_values('billing_date')
train_size = int(len(full_df) * 0.8)
train_df = full_df.iloc[:train_size]
test_df = full_df.iloc[train_size:]

target = 'claim_status'
features = [
    'age', 'gender', 'chronic_flag', 'city', 'department', 'visit_type',
    'length_of_stay_hours', 'billed_amount', 'approved_amount', 'payment_days',
    'visit_frequency', 'avg_length_of_stay', 'provider_rejection_rate',
    'days_since_registration', 'visit_month', 'visit_weekday'
]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# ----------------------------
# Define numeric and categorical columns
# ----------------------------
numeric_features = [
    'age', 'length_of_stay_hours', 'billed_amount', 'approved_amount', 'payment_days',
    'visit_frequency', 'avg_length_of_stay', 'provider_rejection_rate',
    'days_since_registration', 'visit_month', 'visit_weekday'
]
categorical_features = ['gender', 'city', 'department', 'visit_type']

# ----------------------------
# Preprocessing pipelines
# ----------------------------
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # fill missing numeric with median
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing categorical with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# ----------------------------
# Baseline model: Logistic Regression
# ----------------------------
lr_model_claim = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

lr_model_claim.fit(X_train, y_train)
y_pred_claim = lr_model_claim.predict(X_test)
print("Baseline Logistic Regression Report:")
print(classification_report(y_test, y_pred_claim))

# ----------------------------
# Advanced model: Random Forest
# ----------------------------
rf_model_claim = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])

rf_model_claim.fit(X_train, y_train)
y_pred_rf_claim = rf_model_claim.predict(X_test)
print("Random Forest Report:")
print(classification_report(y_test, y_pred_rf_claim))

# ----------------------------
# Save models
# ----------------------------
joblib.dump(lr_model_claim, 'claim_lr_model.pkl')
joblib.dump(rf_model_claim, 'claim_rf_model.pkl')


=== Model B: Claim Outcome Classification ===
Baseline Logistic Regression Report:
              precision    recall  f1-score   support

        Paid       0.95      0.95      0.95      2974
     Pending       0.88      0.89      0.88      1258
    Rejected       0.96      0.94      0.95       768

    accuracy                           0.93      5000
   macro avg       0.93      0.93      0.93      5000
weighted avg       0.93      0.93      0.93      5000

Random Forest Report:
              precision    recall  f1-score   support

        Paid       0.86      0.98      0.92      2974
     Pending       0.92      0.64      0.76      1258
    Rejected       1.00      0.96      0.98       768

    accuracy                           0.89      5000
   macro avg       0.93      0.86      0.88      5000
weighted avg       0.90      0.89      0.89      5000



['claim_rf_model.pkl']

In [ ]:
# Feature Schema

import json
feature_schema = {f: 'numeric' for f in numeric_features}
feature_schema.update({f: 'categorical' for f in categorical_features})
with open('feature_schema.json', 'w') as f:
    json.dump(feature_schema, f, indent=4)

print("Claim models and feature schema saved successfully!")

Claim models and feature schema saved successfully!
